# 02 — ACEA Registration Parsing

This notebook parses the ACEA source files downloaded in the previous step and converts them into structured monthly registration tables.

The raw ACEA files use different formats across years, so this notebook applies multiple extraction approaches:

- `extract_text()` for standard PDF text extraction
- `extract_words()` as a fallback for more complex PDF layouts
- Excel-based extraction for 2022 source files

The final output is a standardized dataset containing monthly new car registrations by market and power source.

In [ ]:
import re
import itertools
import calendar
from pathlib import Path

import pandas as pd
import pdfplumber
import requests

In [ ]:
# ----------------------------
# PATH CONFIGURATION
# ----------------------------

BRONZE_DIR = DATA_DIR / "bronze" / "acea_pdfs"
SILVER_DIR = DATA_DIR / "silver" / "acea_registrations"

In [ ]:
# ------------------------------------------------------------
# 1) LOCATE LOCAL ACEA PDF FILES
# ------------------------------------------------------------
def find_local_pdf(month_label: str, year_label: int) -> Path:
    candidates = [
        BRONZE_DIR / f"Press_release_car_registrations_{month_label}_{year_label}.pdf",
        BRONZE_DIR / f"Press_release_car_registrations-{month_label}_{year_label}.pdf",
        BRONZE_DIR / f"Press_release_car_registrations_{month_label}-{year_label}.pdf",
    ]

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError(f"No local PDF found for {month_label} {year_label}")



In [ ]:
# ------------------------------------------------------------
# 2) EXTRACT_TEXT PARSER
# ------------------------------------------------------------
def extract_pdf_text_by_page(pdf_path: Path) -> list[dict]:
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            pages.append({"page_num": i, "text": text})
    return pages


def detect_template(pages: list[dict]) -> str:
    full_text = "\n".join(p["text"] for p in pages)

    if "NEW CAR REGISTRATIONS BY MARKET AND POWER SOURCE" in full_text:
        return "template_new"
    if "NEW PASSENGER CAR REGISTRATIONS, BY MARKET" in full_text:
        return "template_old"

    return "unknown"


def extract_monthly_total_table_template_new(pages: list[dict], source_file: str) -> pd.DataFrame:
    monthly_page = next(
        (p for p in pages
         if "NEW CAR REGISTRATIONS BY MARKET AND POWER SOURCE" in p["text"]
         and "MONTHLY" in p["text"]),
        None
    )
    if monthly_page is None:
        raise ValueError(f"Monthly table page not found in {source_file}")

    lines = [ln.strip() for ln in monthly_page["text"].splitlines() if ln.strip()]
    return pd.DataFrame({"raw_line": lines})


def extract_monthly_total_table_template_old(pages: list[dict], source_file: str) -> pd.DataFrame:
    page = next(
        (p for p in pages if "NEW PASSENGER CAR REGISTRATIONS, BY MARKET" in p["text"]),
        None
    )
    if page is None:
        raise ValueError(f"Old-style market table page not found in {source_file}")

    lines = [ln.strip() for ln in page["text"].splitlines() if ln.strip()]
    return pd.DataFrame({"raw_line": lines})


def parse_with_extract_text(pdf_path: Path, year_label: int, month_label: str) -> pd.DataFrame:
    pages = extract_pdf_text_by_page(pdf_path)
    template = detect_template(pages)

    if template == "template_new":
        df_raw = extract_monthly_total_table_template_new(pages, pdf_path.name)
    elif template == "template_old":
        df_raw = extract_monthly_total_table_template_old(pages, pdf_path.name)
    else:
        raise ValueError("Unknown template")

    df = df_raw.copy()

    exclude_starts = [
        "NEW CAR REGISTRATIONS",
        "MONTHLY",
        "BATTERY",
        month_label,
        str(year_label),
        "Source:",
        "ACEA",
        "Page",
        "www.acea.auto",
        "1 ",
        "2 ",
    ]

    df = df[
        ~df["raw_line"].str.startswith(tuple(exclude_starts), na=False)
    ].copy()

    df = df[
        ~df["raw_line"].str.fullmatch(r"\d+", na=False)
    ].copy()

    def is_numericish_token(token: str) -> bool:
        token = str(token).strip()
        return bool(re.fullmatch(r"[+\-]?\d[\d,\.]*", token))

    def normalize_country_label(label: str) -> str:
        label = label.strip()
        label = re.sub(r"(?<=[A-Za-z])\d+$", "", label)
        label = re.sub(r"[()]", "", label)
        label = re.sub(r"\s+", " ", label).strip()

        key = label.upper()
        replacements = {
            "EUROPEAN UNION": "European Union",
            "EUROPEAN UNION EU": "European Union",
            "UNITED KINGDOM": "United Kingdom",
            "EU + EFTA + UK": "EU EFTA UK",
            "EU+EFTA+UK": "EU EFTA UK",
            "EU EFTA UK": "EU EFTA UK",
        }
        return replacements.get(key, label.title())

    def split_country_and_values(line: str):
        tokens = line.split()

        label_tokens = []
        value_tokens = []
        numeric_started = False

        for tok in tokens:
            if not numeric_started and not is_numericish_token(tok):
                label_tokens.append(tok)
            else:
                numeric_started = True
                value_tokens.append(tok)

        if not label_tokens:
            return [None]

        country = normalize_country_label(" ".join(label_tokens))
        return [country] + value_tokens

    split_rows = df["raw_line"].apply(split_country_and_values).tolist()
    df_split = pd.DataFrame(split_rows)
    df_split.columns = ["country"] + [f"col_{i}" for i in range(1, df_split.shape[1])]

    df_clean = pd.DataFrame({
        "country": df_split["country"],
        "battery_electric": df_split.get("col_1"),
        "plug_in_hybrid": df_split.get("col_4"),
        "hybrid_electric": df_split.get("col_7"),
        "others": df_split.get("col_10"),
        "petrol": df_split.get("col_13"),
        "diesel": df_split.get("col_16"),
        "total": df_split.get("col_19"),
    })

    bad_rows = {"European", "Eu", "Www.Acea.Auto", "Acea", "Page", "1", "2"}
    df_clean = df_clean[~df_clean["country"].isin(bad_rows)].copy()

    for col in df_clean.columns[1:]:
        df_clean[col] = (
            df_clean[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .replace({"None": pd.NA, "": pd.NA, "nan": pd.NA})
            .pipe(pd.to_numeric, errors="coerce")
            .round(0)
            .astype("Int64")
        )

    return df_clean.reset_index(drop=True)


## Parser 2 — `extract_words()` Fallback

This parser is used when standard text extraction does not produce a valid structured table.

It extracts individual word positions from the PDF, reconstructs rows based on vertical alignment, identifies numeric columns from the table header, and reconciles fuel-type values against the reported total.

In [ ]:
# ------------------------------------------------------------
# 3) EXTRACT_WORDS PARSER
# ------------------------------------------------------------

def parse_with_extract_words(pdf_path: Path, year_label: int) -> pd.DataFrame:

    with pdfplumber.open(pdf_path) as pdf:
        target_page = None

        for page in pdf.pages:
            text = page.extract_text() or ""

            if (
                "MONTHLY" in text
                and "REGISTRATIONS" in text
                and ("POWER SOURCE" in text or "BY MARKET")
            ):
                target_page = page
                break

        if target_page is None:
            raise ValueError(f"Could not find monthly table page in {pdf_path.name}")

        words = target_page.extract_words()

    words_df = pd.DataFrame(words).copy()
    words_df["x_center"] = (words_df["x0"] + words_df["x1"]) / 2
    words_df = words_df.sort_values(["top", "x0"]).reset_index(drop=True)

    def group_words_into_rows(words_df, y_tolerance=3):
        rows = []
        current_row = []
        current_top = None

        for _, word in words_df.iterrows():
            row_word = word.to_dict()
            if current_top is None:
                current_top = word["top"]
                current_row = [row_word]
            elif abs(word["top"] - current_top) <= y_tolerance:
                current_row.append(row_word)
            else:
                rows.append(current_row)
                current_row = [row_word]
                current_top = word["top"]

        if current_row:
            rows.append(current_row)

        return rows

    rows = group_words_into_rows(words_df, y_tolerance=3)

    header_row = None
    for row in rows:
        tokens = [w["text"] for w in sorted(row, key=lambda x: x["x0"])]
        year_tokens = [t for t in tokens if str(t).startswith(str(year_label))]
        if len(year_tokens) >= 7:
            header_row = sorted(row, key=lambda x: x["x0"])
            break

    if header_row is None:
        raise ValueError(f"Could not find header row with {year_label} labels.")

    current_year_tokens = [w for w in header_row if str(w["text"]).startswith(str(year_label))][:7]

    if len(current_year_tokens) != 7:
        raise ValueError(f"Expected 7 current-year header tokens, found {len(current_year_tokens)}")

    col_centers = [(w["x0"] + w["x1"]) / 2 for w in current_year_tokens]

    target_cols = [
        "battery_electric",
        "plug_in_hybrid",
        "hybrid_electric",
        "others",
        "petrol",
        "diesel",
        "total"
    ]

    def row_tokens(row):
        return [w["text"] for w in sorted(row, key=lambda x: x["x0"])]

    def is_data_row(row):
        txt = " ".join(row_tokens(row))
        skip_patterns = [
            "NEW CAR REGISTRATIONS", "MONTHLY", "BATTERY", "PLUG", "HYBRID",
            "PETROL", "DIESEL", "TOTAL", "Includes", "Page",
            str(year_label), str(year_label - 1), f"{str(year_label)[-2:]}/{str(year_label - 1)[-2:]}"
        ]
        return not any(p in txt for p in skip_patterns)

    def looks_like_country_piece(text):
        return bool(re.fullmatch(r"[A-Za-z]+", str(text)))

    def clean_country(parts):
        country = "".join(parts)

        # Normalize aggregate names often concatenated by extract_words
        replacements = {
            "EUROPEANUNION": "European Union",
            "UnitedKingdom": "United Kingdom",
            "EUEFTAUK": "EU EFTA UK",
        }
        return replacements.get(country, country)

    def extract_first_valid_number(text, col_name):
        text = str(text)

        m = re.search(r"\d{1,3}(?:,\d{3})+", text)
        if m:
            return m.group(0).replace(",", "")

        m = re.search(r"\d+", text)
        if not m:
            return None

        digits = m.group(0)

        if col_name != "total" and len(digits) > 5:
            return digits[:5]
        if col_name == "total" and len(digits) > 7:
            return digits[:7]

        return digits

    records = []

    for row in rows:
        if not is_data_row(row):
            continue

        row_sorted = sorted(row, key=lambda x: x["x0"])

        country_parts = []
        for w in row_sorted:
            x_center = (w["x0"] + w["x1"]) / 2
            if x_center < col_centers[0] - 15 and looks_like_country_piece(w["text"]):
                country_parts.append(w["text"])

        country = clean_country(country_parts)
        if not country:
            continue

        record = {"country": country}

        for col_name, center in zip(target_cols, col_centers):
            candidates = []
            for w in row_sorted:
                x_center = (w["x0"] + w["x1"]) / 2
                if abs(x_center - center) <= 18:
                    cleaned = extract_first_valid_number(w["text"], col_name)
                    if cleaned is not None:
                        candidates.append((abs(x_center - center), cleaned, w["text"]))

            if candidates:
                candidates = sorted(candidates, key=lambda x: x[0])
                record[col_name] = candidates[0][1]
            else:
                record[col_name] = None

        records.append(record)

    df_clean = pd.DataFrame(records)
    df_clean = df_clean[df_clean["country"].str.len() > 2].reset_index(drop=True)

    for col in target_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").astype("Int64")

    fuel_cols = [
        "battery_electric",
        "plug_in_hybrid",
        "hybrid_electric",
        "others",
        "petrol",
        "diesel",
    ]

    def candidate_values(value, col_name):
        if pd.isna(value):
            return [pd.NA]

        s = str(int(value))
        candidates = {int(s)}

        if len(s) >= 5:
            candidates.add(int(s[:3]))
        if len(s) >= 6:
            candidates.add(int(s[:4]))
        if len(s) >= 7:
            candidates.add(int(s[:5]))
        if len(s) >= 8:
            candidates.add(int(s[:6]))

        if col_name == "others":
            candidates.add(0)

        return sorted(candidates)

    def reconcile_row_keep_na(row):
        if pd.isna(row["total"]):
            return row

        total = int(row["total"])

        candidate_map = {}
        for col in fuel_cols:
            candidate_map[col] = candidate_values(row[col], col)

        variable_cols = []
        fixed_values = {}

        for col in fuel_cols:
            val = row[col]

            if pd.isna(val):
                variable_cols.append(col)
                continue

            val = int(val)

            if val > total or len(str(val)) >= 5:
                variable_cols.append(col)
            else:
                fixed_values[col] = val

        if not variable_cols:
            return row

        candidate_lists = [candidate_map[col] for col in variable_cols]

        best_combo = None
        best_diff = float("inf")

        for combo in itertools.product(*candidate_lists):
            test_vals = fixed_values.copy()

            for col, val in zip(variable_cols, combo):
                test_vals[col] = val

            row_sum = sum(0 if pd.isna(v) else int(v) for v in test_vals.values())
            diff = abs(row_sum - total)

            if diff < best_diff:
                best_diff = diff
                best_combo = test_vals.copy()

        for col in fuel_cols:
            row[col] = best_combo.get(col, pd.NA)

        return row

    df_clean = df_clean.apply(reconcile_row_keep_na, axis=1)

    # infer exactly one missing value only
    for idx, row in df_clean.iterrows():
        total = row["total"]
        if pd.isna(total):
            continue

        missing_cols = [col for col in fuel_cols if pd.isna(row[col])]
        if len(missing_cols) == 1:
            known_sum = sum(int(row[col]) for col in fuel_cols if not pd.isna(row[col]))
            inferred = int(total) - known_sum

            if inferred >= 0:
                df_clean.at[idx, missing_cols[0]] = inferred

    for col in target_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").astype("Int64")

    return df_clean



## Parsed Output Validation

This validation function checks whether the parsed table is structurally plausible before accepting it.

The checks are designed to catch common PDF extraction issues, such as missing countries, incomplete numeric columns, or accidentally extracting percentage fragments instead of registration volumes.

In [ ]:
# ------------------------------------------------------------
# 4) VALIDATION
# ------------------------------------------------------------
def validate_parsed_output(df: pd.DataFrame) -> bool:
    if df is None or df.empty:
        return False

    if len(df) < 25:
        return False

    countries_present = set(df["country"].dropna())

    # Must contain core countries
    expected_countries = {"Austria", "France", "Germany"}
    if not expected_countries.issubset(countries_present):
        return False

    # Must contain critical aggregates for the model
    required_aggregates = {"European Union", "United Kingdom"}
    if not required_aggregates.issubset(countries_present):
        return False

    if "battery_electric" in df.columns:
        if df["battery_electric"].notna().sum() < 20:
            return False

    if "total" in df.columns:
        if df["total"].notna().sum() < 20:
            return False

    # European Union total should be a real positive volume, not a percentage fragment
    eu_row = df[df["country"] == "European Union"]
    if eu_row.empty or eu_row["total"].isna().all():
        return False

    eu_total = eu_row["total"].dropna().iloc[0]
    if eu_total < 100000:
        return False

    # United Kingdom total should also be plausible
    uk_row = df[df["country"] == "United Kingdom"]
    if uk_row.empty or uk_row["total"].isna().all():
        return False

    uk_total = uk_row["total"].dropna().iloc[0]
    if uk_total < 20000:
        return False

    return True




## Master Extraction Logic

This function applies a two-step extraction strategy:

1. Attempt standard PDF text extraction (`extract_text`)
2. Validate the output
3. If validation fails, fall back to a more robust word-based extraction (`extract_words`)

This ensures consistent and reliable parsing across different ACEA report formats and layouts.

In [ ]:
# ------------------------------------------------------------
# 5) MASTER EXTRACTION: text first, words fallback
# ------------------------------------------------------------
def extract_with_text_first(pdf_path: Path, year_label: int, month_label: str) -> pd.DataFrame:
    try:
        df_text = parse_with_extract_text(pdf_path, year_label, month_label)

        if validate_parsed_output(df_text):
            print(f"{pdf_path.name}: using extract_text()")
            df_text["extraction_method"] = "extract_text"
            return df_text
        else:
            print(f"{pdf_path.name}: extract_text() failed validation")

    except Exception as e:
        print(f"{pdf_path.name}: extract_text() failed -> {e}")

    print(f"{pdf_path.name}: falling back to extract_words()")
    df_words = parse_with_extract_words(pdf_path, year_label)

    if not validate_parsed_output(df_words):
        raise ValueError(f"{pdf_path.name}: extract_words() also failed validation")

    df_words["extraction_method"] = "extract_words"
    return df_words




## Run Parsing for Full Year

This step applies the parsing logic across all monthly ACEA reports for a given year.

For each month:
- the corresponding PDF file is located in the bronze layer
- the extraction logic is applied
- the output is enriched with time metadata (year, month)


In [ ]:
# ------------------------------------------------------------
# 6) FIND LOCAL PDF + RUN YEAR
# ------------------------------------------------------------
def find_local_pdf(month_label: str, year_label: int) -> Path:
    candidates = [
        BRONZE_DIR / f"Press_release_car_registrations_{month_label}_{year_label}.pdf",
        BRONZE_DIR / f"Press_release_car_registrations-{month_label}_{year_label}.pdf",
        BRONZE_DIR / f"Press_release_car_registrations_{month_label}-{year_label}.pdf",
    ]

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError(f"No local PDF found for {month_label} {year_label}")

def run_pdf_year(year_label: int) -> pd.DataFrame:
    results = []

    for month_num in range(1, 13):
        month_label = calendar.month_name[month_num]

        try:
            pdf_path = find_local_pdf(month_label, year_label)
            df_month = extract_with_text_first(pdf_path, year_label, month_label)

            df_month["year"] = year_label
            df_month["month"] = month_num
            df_month["month_name"] = month_label

            results.append(df_month)

        except Exception as e:
            print(f"{month_label} {year_label} failed: {e}")

    if results:
        return pd.concat(results, ignore_index=True)
    else:
        return pd.DataFrame()

In [ ]:
df_2023_pdf = run_pdf_year(2023)
df_2024_pdf = run_pdf_year(2024)
df_2025_pdf = run_pdf_year(2025)

In [ ]:
# ------------------------------------------------------------
# 7) 2023 cleanup
# ------------------------------------------------------------

def clean_pdf_rows(df: pd.DataFrame) -> pd.DataFrame:
    bad_country_prefixes = (
        "New Passenger Car Registrations",
        "New Car Registrations",
        "Source:",
        "Page",
        "www.",
    )

    bad_exact_countries = {
        "1",
        "2",
        "NEW",
        "Acea",
        "Www.Acea.Auto",
    }

    df = df.copy()

    df = df[
        ~df["country"].astype(str).str.startswith(bad_country_prefixes, na=False)
    ]

    df = df[
        ~df["country"].astype(str).isin(bad_exact_countries)
    ]

    return df.reset_index(drop=True)

In [ ]:
df_2023_pdf = clean_pdf_rows(df_2023_pdf)
df_2024_pdf = clean_pdf_rows(df_2024_pdf)
df_2025_pdf = clean_pdf_rows(df_2025_pdf)

# Extracting 2022 data

## Parse 2022 Excel-Based ACEA Data

The 2022 ACEA data uses a different source structure from later years. Monthly total registrations are taken from monthly Excel files, while fuel-type breakdowns are available at quarterly level.

To align 2022 with the monthly structure used in the rest of the project, quarterly fuel values are distributed evenly across the three months of each quarter.

In [ ]:
import re
import calendar
from pathlib import Path

import pandas as pd


# ------------------------------------------------------------
# 1) HELPERS
# ------------------------------------------------------------
def normalize_country_2022(name: str) -> str:
    if pd.isna(name):
        return name

    name = str(name).strip()
    name = re.sub(r"\d+$", "", name).strip()

    replacements = {
        "Czech Republic": "Czechia",
        "EUROPEAN UNION": "European Union",
        "United Kingdom": "United Kingdom",
        "EU + EFTA + UK": "EU EFTA UK",
        "EU14": "EU14",
        "EU12": "EU12",
        "EFTA": "EFTA",
    }

    return replacements.get(name, name)


def quarter_from_month(month_num: int) -> int:
    return ((month_num - 1) // 3) + 1


# ------------------------------------------------------------
# 2) LOCATE 2022 MONTHLY TOTAL FILE
#    Example filename:
#    Press_release_car_registrations_January_2022.xlsx
# ------------------------------------------------------------
def find_2022_monthly_excel(month_label: str) -> Path:
    candidates = [
        BRONZE_DIR / f"Press_release_car_registrations_{month_label}_2022.xlsx",
        BRONZE_DIR / f"Press_release_car_registrations-{month_label}_2022.xlsx",
        BRONZE_DIR / f"Press_release_car_registrations_{month_label}-2022.xlsx",
    ]

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError(f"No 2022 monthly Excel found for {month_label}")


# ------------------------------------------------------------
# 3) PARSE ONE 2022 MONTHLY TOTAL FILE
#    Uses Market sheet, column 1 = country, column 2 = 2022 units
# ------------------------------------------------------------
def parse_2022_monthly_total(xlsx_path: Path, month_num: int, month_label: str) -> pd.DataFrame:
    raw = pd.read_excel(xlsx_path, sheet_name="Market", header=None)

    df = raw[[1, 2]].copy()
    df.columns = ["country", "total"]

    df = df[df["country"].notna()].copy()
    df["country"] = df["country"].apply(normalize_country_2022)

    # Keep only rows where total is numeric
    df["total"] = pd.to_numeric(df["total"], errors="coerce")
    df = df[df["total"].notna()].copy()

    # Remove footer/source rows
    bad_prefixes = (
        "SOURCE:",
        "Member states",
    )
    df = df[~df["country"].astype(str).str.startswith(bad_prefixes, na=False)].copy()

    df["total"] = df["total"].astype("Int64")
    df["year"] = 2022
    df["month"] = month_num
    df["month_name"] = month_label
    df["quarter"] = quarter_from_month(month_num)

    return df.reset_index(drop=True)


# ------------------------------------------------------------
# 4) RUN ALL 12 MONTHLY TOTAL FILES
# ------------------------------------------------------------
def run_2022_monthly_totals() -> pd.DataFrame:
    results = []

    for month_num in range(1, 13):
        month_label = calendar.month_name[month_num]
        xlsx_path = find_2022_monthly_excel(month_label)
        df_month = parse_2022_monthly_total(xlsx_path, month_num, month_label)
        results.append(df_month)

    return pd.concat(results, ignore_index=True)


# ------------------------------------------------------------
# 5) LOCATE QUARTERLY FUEL FILE
#    Example sample:
#    20220505_PRPC-fuel_Q1-2022_FINAL.xlsx
# ------------------------------------------------------------
def find_2022_quarterly_fuel_file(q: int) -> Path:
    candidates = list(BRONZE_DIR.glob(f"*fuel_Q{q}-2022*.xlsx"))

    if not candidates:
        raise FileNotFoundError(f"No quarterly fuel workbook found for Q{q} 2022")

    return candidates[0]


# ------------------------------------------------------------
# 6) PARSE ONE FUEL SHEET FROM A QUARTERLY WORKBOOK
#    Sheets:
#    BEV, PHEV, HEV, NGV, LPG + Other, Petrol, Diesel
#    country in col 1, 2022 units in col 2
# ------------------------------------------------------------
def parse_fuel_sheet(xlsx_path: Path, sheet_name: str) -> pd.DataFrame:
    raw = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None)

    df = raw[[1, 2]].copy()
    df.columns = ["country", "units_q"]

    df = df[df["country"].notna()].copy()
    df["country"] = df["country"].apply(normalize_country_2022)

    df["units_q"] = pd.to_numeric(df["units_q"], errors="coerce")
    df = df[df["units_q"].notna()].copy()

    # Remove source/footer rows
    bad_prefixes = (
        "SOURCE:",
        "Member states",
    )
    df = df[~df["country"].astype(str).str.startswith(bad_prefixes, na=False)].copy()

    return df.reset_index(drop=True)


# ------------------------------------------------------------
# 7) PARSE ONE QUARTERLY FUEL WORKBOOK
#    "others" = NGV + LPG + Other
#    Then convert quarter totals into average monthly values
# ------------------------------------------------------------
def parse_2022_quarterly_fuel(q: int) -> pd.DataFrame:
    xlsx_path = find_2022_quarterly_fuel_file(q)

    bev = parse_fuel_sheet(xlsx_path, "BEV").rename(columns={"units_q": "battery_electric_q"})
    phev = parse_fuel_sheet(xlsx_path, "PHEV").rename(columns={"units_q": "plug_in_hybrid_q"})
    hev = parse_fuel_sheet(xlsx_path, "HEV").rename(columns={"units_q": "hybrid_electric_q"})
    ngv = parse_fuel_sheet(xlsx_path, "NGV").rename(columns={"units_q": "ngv_q"})
    lpg = parse_fuel_sheet(xlsx_path, "LPG + Other").rename(columns={"units_q": "lpg_other_q"})
    petrol = parse_fuel_sheet(xlsx_path, "Petrol").rename(columns={"units_q": "petrol_q"})
    diesel = parse_fuel_sheet(xlsx_path, "Diesel").rename(columns={"units_q": "diesel_q"})

    df = bev.merge(phev, on="country", how="outer") \
            .merge(hev, on="country", how="outer") \
            .merge(ngv, on="country", how="outer") \
            .merge(lpg, on="country", how="outer") \
            .merge(petrol, on="country", how="outer") \
            .merge(diesel, on="country", how="outer")

    df["others_q"] = df[["ngv_q", "lpg_other_q"]].fillna(0).sum(axis=1)

    df = df[[
        "country",
        "battery_electric_q",
        "plug_in_hybrid_q",
        "hybrid_electric_q",
        "others_q",
        "petrol_q",
        "diesel_q",
    ]].copy()

    # average monthly values for the quarter
    for col in df.columns[1:]:
        df[col.replace("_q", "")] = df[col] / 3

    df = df[[
    "country",
    "battery_electric",
    "plug_in_hybrid",
    "hybrid_electric",
    "others",
    "petrol",
    "diesel",
]].copy()

    for col in [
        "battery_electric",
        "plug_in_hybrid",
        "hybrid_electric",
        "others",
        "petrol",
        "diesel",
]:
        df[col] = df[col].fillna(0).astype(int).astype("Int64")

    df["quarter"] = q
    
    return df



# ------------------------------------------------------------
# 8) EXPAND QUARTERLY FUEL DATA TO MONTHS
#    Q1 -> Jan/Feb/Mar, etc.
# ------------------------------------------------------------
def expand_2022_quarterly_fuel_to_months() -> pd.DataFrame:
    quarter_months = {
        1: [1, 2, 3],
        2: [4, 5, 6],
        3: [7, 8, 9],
        4: [10, 11, 12],
    }

    results = []

    for q in [1, 2, 3, 4]:
        df_q = parse_2022_quarterly_fuel(q)

        for m in quarter_months[q]:
            df_m = df_q.copy()
            df_m["year"] = 2022
            df_m["month"] = m
            df_m["month_name"] = calendar.month_name[m]
            results.append(df_m)

    return pd.concat(results, ignore_index=True)


# ------------------------------------------------------------
# 9) FINAL 2022 DATASET = MONTHLY TOTALS + MONTHLY AVG FUELS
# ------------------------------------------------------------
def build_2022_dataset() -> pd.DataFrame:
    monthly_totals = run_2022_monthly_totals()
    monthly_fuels = expand_2022_quarterly_fuel_to_months()

    df_2022 = monthly_totals.merge(
        monthly_fuels,
        on=["country", "year", "month", "month_name", "quarter"],
        how="left"
    )

    df_2022["extraction_method"] = "excel_monthly_total + quarterly_fuel_avg"

    df_2022 = df_2022[
        [
            "country",
            "battery_electric",
            "plug_in_hybrid",
            "hybrid_electric",
            "others",
            "petrol",
            "diesel",
            "total",
            "extraction_method",
            "year",
            "month",
            "month_name",
            "quarter",
        ]
    ].copy()

    return df_2022

In [ ]:
%pip install openpyxl

## Consolidate Parsed ACEA Data Across Years

This section combines the parsed 2022 Excel-based data with the parsed 2023–2025 PDF data into one standardized dataset.

In [ ]:
# ------------------------------------------------------------
# 1) ADD QUARTER TO PDF YEARS
# ------------------------------------------------------------
for df in [df_2023_pdf, df_2024_pdf, df_2025_pdf]:
    df["quarter"] = ((df["month"] - 1) // 3) + 1


# ------------------------------------------------------------
# 2) STANDARDIZE COLUMN ORDER
# ------------------------------------------------------------
final_cols = [
    "country",
    "battery_electric",
    "plug_in_hybrid",
    "hybrid_electric",
    "others",
    "petrol",
    "diesel",
    "total",
    "extraction_method",
    "year",
    "month",
    "month_name",
    "quarter",
]

# make sure all dfs have the same columns
df_2022 = df_2022[final_cols].copy()
df_2023_pdf   = df_2023_pdf[final_cols].copy()
df_2024_pdf   = df_2024_pdf[final_cols].copy()
df_2025_pdf   = df_2025_pdf[final_cols].copy()


# ------------------------------------------------------------
# 3) CONCATENATE ALL YEARS
# ------------------------------------------------------------
df_all = pd.concat(
    [df_2022, df_2023_pdf, df_2024_pdf, df_2025_pdf],
    ignore_index=True
)


# ------------------------------------------------------------
# 4) OPTIONAL: STANDARDIZE DTYPES
# ------------------------------------------------------------
numeric_cols = [
    "battery_electric",
    "plug_in_hybrid",
    "hybrid_electric",
    "others",
    "petrol",
    "diesel",
    "total",
    "year",
    "month",
    "quarter",
]

for col in numeric_cols:
    df_all[col] = pd.to_numeric(df_all[col], errors="coerce").astype("Int64")


# ------------------------------------------------------------
# 5) QUICK SANITY CHECKS
# ------------------------------------------------------------
print(df_all.shape)
print(df_all.columns.tolist())

print("\nRows per year:")
print(df_all["year"].value_counts().sort_index())

print("\nMissing values:")
print(df_all[["battery_electric", "total"]].isna().sum())

print("\nSample:")
display(df_all.head(20))

In [ ]:
df_filtered = df_all[df_all["month_name"] == "August"]


In [ ]:
# check European Union across time
display(
    df_all[df_all["country"] == "European Union"]
    .sort_values(["year", "month"])[["country", "year", "month", "quarter", "battery_electric", "total"]]
    .head(20)
)

In [ ]:
# check one normal country across time
display(
    df_all[df_all["country"] == "France"]
    .sort_values(["year", "month"])[["country", "year", "month", "quarter", "battery_electric", "total"]]
    
)

In [ ]:
country_to_code_mapping = {
    "Austria": "AT",
    "Belgium": "BE",
    "Bulgaria": "BG",
    "Cyprus": "CY",
    "Czechia": "CZ",
    "Czech Republic": "CZ",
    "CzechRepublic": "CZ",
    "Germany": "DE",
    "Denmark": "DK",
    "Estonia": "EE",
    "European Union": "EU",
    "Euro Zone": "EUR",
    "Spain": "ES",
    "Finland": "FI",
    "France": "FR",
    "Greece": "GR",
    "Croatia": "HR",
    "Hungary": "HU",
    "Iceland": "IS",
    "Ireland": "IE",
    "Italy": "IT",
    "Lithuania": "LT",
    "Luxembourg": "LU",
    "Latvia": "LV",
    "Malta": "MT",
    "Netherlands": "NL",
    "Norway": "NO",
    "Switzerland": "CH",
    "United Kingdom": "UK",
    "Poland": "PL",
    "Portugal": "PT",
    "Romania": "RO",
    "Sweden": "SE",
    "Slovenia": "SI",
    "Slovakia": "SK"
}

In [ ]:
df_all["country_code"] = df_all["country"].map(country_to_code_mapping)

In [ ]:
acea_df_all = df_all.copy()

In [ ]:
# =========================================================
# DROP NON-COUNTRY / AGGREGATE ROWS FROM ACEA
# =========================================================

drop_countries = {
    "EU",
    "EFTA",
    "Efta",
    "EU EFTA UK",
    "EU14 + EFTA + UK",
    "Sep"
}

acea_df_all = acea_df_all[
    ~acea_df_all["country"].isin(drop_countries)
].copy()

In [ ]:

# ------------------------------------------------------------
# SAVE TO SILVER LAYER
# ------------------------------------------------------------
output_path = SILVER_DIR / "acea_registrations_2022_2025.parquet"

acea_df_all.to_parquet(output_path, index=False)

print(f"Saved parsed ACEA dataset to: {output_path}")

In [ ]:
display(acea_df_all["country"].unique())

In [ ]:
display(acea_df_all["country_code"].unique())